# UPI Fraud Detection — Feature Engineering
**Author:** Umesh R Kale | EDXcellence Internship

This notebook walks through every feature we build and explains the real-world fraud signal behind it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.append('..')
warnings_import = __import__('warnings'); warnings_import.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/upi_transactions.csv', parse_dates=['timestamp'])
print(f'Loaded {len(df):,} transactions')

## 1. Time Features
Fraudsters prefer late nights when victims are asleep and bank support is minimal.

In [ ]:
df['hour']             = df['timestamp'].dt.hour
df['day_of_week']      = df['timestamp'].dt.dayofweek
df['is_weekend']       = (df['day_of_week'] >= 5).astype(int)
df['is_night']         = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)
df['is_midnight']      = ((df['hour'] >= 1) & (df['hour'] <= 4)).astype(int)
df['is_business_hours']= ((df['hour'] >= 9) & (df['hour'] <= 18) & (df['is_weekend']==0)).astype(int)

print('Fraud rate during midnight (1-4 AM):', df[df.is_midnight==1].is_fraud.mean().round(4))
print('Fraud rate during business hours:   ', df[df.is_business_hours==1].is_fraud.mean().round(4))

## 2. Amount Features
A sudden spike in amount is the strongest single fraud signal.

In [ ]:
df['amount_vs_avg_ratio'] = (df['amount'] / df['avg_user_amount'].replace(0, 1)).round(4)
df['is_high_amount']      = (df['amount_vs_avg_ratio'] > 5).astype(int)
df['is_micro_txn']        = (df['amount'] < 10).astype(int)
df['is_round_amount']     = (df['amount'] % 100 == 0).astype(int)
df['log_amount']          = np.log1p(df['amount'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='amount_vs_avg_ratio', hue='is_fraud',
             bins=50, ax=axes[0], palette={0:'steelblue',1:'crimson'})
axes[0].set_xlim(0, 25)
axes[0].set_title('Amount vs Average Ratio — Fraud vs Normal')
axes[0].axvline(5, color='black', linestyle='--', label='5× threshold')
axes[0].legend()

sns.histplot(data=df, x='log_amount', hue='is_fraud',
             bins=40, ax=axes[1], palette={0:'steelblue',1:'crimson'})
axes[1].set_title('Log Amount Distribution')
plt.tight_layout()
plt.show()
print('High amount flag — fraud rate:', df[df.is_high_amount==1].is_fraud.mean().round(4))

## 3. Merchant Features
'Unknown' merchant is nearly always fraud in our dataset.

In [ ]:
merchant_risk = {'grocery':0,'restaurant':0,'pharmacy':0,'utility':1,
                 'fuel':1,'clothing':1,'entertainment':2,'travel':2,'electronics':3,'unknown':5}
df['merchant_risk_score'] = df['merchant_type'].map(merchant_risk).fillna(3)
df['new_merchant_high_amt'] = ((df['is_new_merchant']==1) & (df['amount']>5000)).astype(int)

print('Merchant risk vs fraud rate:')
print(df.groupby('merchant_risk_score')['is_fraud'].agg(['mean','sum','count']).round(3))

## 4. Behaviour Combos
Combination features capture multi-signal fraud patterns.

In [ ]:
df['device_change_high_amt'] = ((df['device_change']==1) & (df['amount']>3000)).astype(int)
df['night_new_merchant']     = ((df['is_night']==1) & (df['is_new_merchant']==1)).astype(int)
df['midnight_high_amount']   = ((df['is_midnight']==1) & (df['amount']>2000)).astype(int)

combos = {
    'midnight_high_amount':   df[df.midnight_high_amount==1].is_fraud.mean(),
    'device_change_high_amt': df[df.device_change_high_amt==1].is_fraud.mean(),
    'night_new_merchant':     df[df.night_new_merchant==1].is_fraud.mean(),
    'new_merchant_high_amt':  df[df.new_merchant_high_amt==1].is_fraud.mean(),
    'is_high_amount':         df[df.is_high_amount==1].is_fraud.mean(),
}

for combo, rate in sorted(combos.items(), key=lambda x: -x[1]):
    bar = '█' * int(rate * 40)
    print(f'  {combo:<30} fraud rate={rate:.3f}  {bar}')

## 5. Final Feature Summary

In [ ]:
feature_cols = [
    'hour','day_of_week','is_weekend','is_night','is_midnight','is_business_hours',
    'amount','log_amount','amount_vs_avg_ratio','is_high_amount','is_micro_txn',
    'is_round_amount','merchant_risk_score','is_new_merchant','new_merchant_high_amt',
    'device_change','device_change_high_amt','night_new_merchant','midnight_high_amount',
]

print(f'Total features: {len(feature_cols)}')
print('\nFraud vs Normal means for key features:')
for col in ['amount_vs_avg_ratio','is_high_amount','is_midnight',
            'merchant_risk_score','midnight_high_amount']:
    if col in df.columns:
        f = df[df.is_fraud==1][col].mean()
        n = df[df.is_fraud==0][col].mean()
        print(f'  {col:<30}  fraud={f:.3f}  normal={n:.3f}  ratio={f/max(n,0.001):.1f}×')